In [2]:
from pathlib import Path

PROJECT = Path.cwd().parent
CLIPS = PROJECT / "data" / "clips"
print(PROJECT)
from pathlib import Path
PROJECT = Path.cwd().parent

c:\Users\seree\OneDrive\Desktop\FalseNine


In [3]:
%pip install -q streamlit imageio-ffmpeg

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os, shutil, json, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd

PROJECT = Path.cwd().parent
ASSETS = PROJECT / "app" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

def find_first(name):
    """Search the project (skipping data/ and app/) for a file called `name`; the newest one wins."""
    hits = []
    for root, dirs, files in os.walk(PROJECT):
        dirs[:] = [d for d in dirs if d not in ("data", "app", ".git", "__pycache__", ".ipynb_checkpoints")]
        if name in files: hits.append(Path(root) / name)
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

FILES = [
    "results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "comparison_test_split.csv",      # 03
    "tracker_comparison.csv", "tracking_summary.png", "track_contact_sheet.png",                            # 04
    "gt_frame_vs_pitch.png", "pitch_view_ours_vs_gt.png",                                                   # 05
    "camera_tracking_drift.csv", "camera_tracking_drift.png",
    "camera_tracking_variants.csv", "camera_tracking_variants.png",                                         # 05b
    "player_workload.csv", "shape_home.csv", "shape_away.csv",
    "distance_per_player.png", "team_shape_over_time.png",                                                  # 06
    "shape_by_possession.csv", "shape_by_possession.png", "space_control.csv", "space_control.png",
    "shape_after_turnover.csv", "shape_after_turnover.png",                                                 # 07
]
missing = []
for name in FILES:
    p = find_first(name)
    if p is None: missing.append(name); continue
    shutil.copy2(p, ASSETS / name)
print("copied:", len(FILES) - len(missing), "of", len(FILES))
print("missing:", missing)

copied: 24 of 24
missing: []


In [5]:
import imageio_ffmpeg
src = find_first("tracked.mp4")
dst = ASSETS / "tracked_h264.mp4"
try:
    subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-i", str(src), "-vf", "scale=1280:-2",
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "26", "-an", str(dst)],
                   check=True, capture_output=True)
    print("saved", dst.name, round(dst.stat().st_size / 1e6, 1), "MB")
except subprocess.CalledProcessError as e:
    print(e.stderr.decode()[-600:])

saved tracked_h264.mp4 1.2 MB


In [6]:
sys.path.append(str(PROJECT / "src"))
import metrica_io as mio

G = mio.load_game(PROJECT / "data" / "metrica", 1)
game = G.tracking
in_play = mio.in_play_flag(game.index, G.events)
poss = mio.possession_flag(game.index, G.events, in_play)

cols = [c for c in game.columns if c.endswith("_x") or c.endswith("_y")] + ["Period", "Time [s]"]
small = game[cols].iloc[::5].round(2).copy()
small["possession"] = poss.reindex(small.index)
small.to_csv(ASSETS / "metrica_game1_5hz.csv.gz", compression="gzip")
print(small.shape, round((ASSETS / "metrica_game1_5hz.csv.gz").stat().st_size / 1e6, 1), "MB")

(29002, 61) 3.1 MB


In [7]:
KEY = {
    "headline": {"map50": 0.926, "false_boxes_before": 61, "false_boxes_after": 2,
                 "calib_error_known_m": 0.31, "in_play_pct": 58},
    "detection": {
        "test_images": 19, "map50": 0.926, "map50_95": 0.607,
        "ap50": {"Ball": 0.810, "Player": 0.972, "Ref": 0.995},
        "before_after": [
            {"model": "Pretrained YOLO11m", "ball found": "13 / 17", "ball extra boxes": 8,
             "people found": "237 / 238", "people extra boxes": 61},
            {"model": "Fine-tuned (FalseNine)", "ball found": "13 / 17", "ball extra boxes": 2,
             "people found": "233 / 238", "people extra boxes": 2}]},
    "tracking": {"frames": 150, "ids_default": 43, "ids_best": 33, "short_default": 27, "short_best": 15,
                 "ball_detected_pct": 74, "ball_filled_pct": 94},
    "pitch": {"homography_residual_m": 0.09, "boxes_matched_pct": 89, "known_median_m": 0.31, "known_p90_m": 0.85,
              "cam_under_1m_frames": 270, "cam_median_m_300_449": 2.7, "cam_error_m_750": 2.1},
}
(ASSETS / "key_results.json").write_text(json.dumps(KEY, indent=1))
print("saved key_results.json")

saved key_results.json


In [8]:
import json, numpy as np, pandas as pd
SITE = PROJECT / "app" / "site"; SITE.mkdir(parents=True, exist_ok=True)

def clean(obj):                       # NaN -> null, round numbers
    return json.loads(pd.DataFrame(obj).round(3).to_json(orient="values"))

def tbl(name, index=False):
    p = ASSETS / name
    if not p.exists(): print("missing:", name); return None
    df = pd.read_csv(p, index_col=0 if index else None)
    if index: df = df.reset_index()
    return {"columns": [str(c) for c in df.columns], "rows": clean(df)}

# drift curve
dr = pd.read_csv(ASSETS / "camera_tracking_drift.csv", index_col=0)
drift = {"x": dr.index.tolist()}
for key, col in [("median", "median_err"), ("p90", "p90_err")]:
    if col in dr: drift[key] = json.loads(dr[col].round(3).to_json(orient="values"))

# team shape, smoothed over 30 s and sampled every 5 s
shape = {"home": {}, "away": {}}
sh = {t: pd.read_csv(ASSETS / f"shape_{t}.csv", index_col=0) for t in ("home", "away")}
metrics = [m for m in ["length", "width", "cx", "line_height", "block_depth", "spread"]
           if m in sh["home"] and m in sh["away"]]
for t in sh:
    sm = sh[t][metrics].rolling(750, min_periods=250).mean().iloc[::125]
    for m in metrics: shape[t][m] = json.loads(sm[m].round(2).to_json(orient="values"))
shape["t"] = (sm.index / 25 / 60).round(2).tolist()

DATA = {
    "key": KEY,
    "tables": {"tracker": tbl("tracker_comparison.csv", True), "variants": tbl("camera_tracking_variants.csv", True),
               "possession": tbl("shape_by_possession.csv"), "space": tbl("space_control.csv"),
               "turnover": tbl("shape_after_turnover.csv")},
    "drift": drift, "shape": shape,
    "workload": json.loads(pd.read_csv(ASSETS / "player_workload.csv").round(3).to_json(orient="records")),
}
(SITE / "data.js").write_text("window.DATA=" + json.dumps(DATA) + ";")

# replay: every 10th frame (2.5 Hz), positions in decimetres, missing = -9999
STEP = 10
sub = game.iloc[::STEP]
ids = lambda side: sorted({c.split("_")[1] for c in game.columns if c.startswith(side + "_") and c.endswith("_x")}, key=int)
hid, aid = ids("Home"), ids("Away")
def dm(cols): return np.rint(sub[cols].fillna(-999.9).to_numpy() * 10).astype(int)
code = poss.reindex(sub.index).map({"none": 0, "Home": 1, "Away": 2, "dead": 3}).fillna(0).astype(int).to_numpy()
M = np.column_stack([np.rint(sub["Time [s]"].to_numpy()).astype(int), sub["Period"].astype(int), code,
                     dm(["ball_x"]), dm(["ball_y"]),
                     dm([f"Home_{i}_x" for i in hid]), dm([f"Home_{i}_y" for i in hid]),
                     dm([f"Away_{i}_x" for i in aid]), dm([f"Away_{i}_y" for i in aid])])
R = {"stride": M.shape[1], "n_home": len(hid), "n_away": len(aid), "hz": 25 / STEP,
     "t_end": float(sub["Time [s]"].iloc[-1]), "data": M.ravel().tolist()}
(SITE / "replay.js").write_text("window.REPLAY=" + json.dumps(R) + ";")
print("home", len(hid), "away", len(aid), "frames", len(M), "| replay.js MB:", round((SITE / "replay.js").stat().st_size / 1e6, 1))

home 14 away 14 frames 14501 | replay.js MB: 4.8
